In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
# Read the required data
train_data = pd.read_csv('data/train.csv')

X_train = train_data.drop(columns=['id', 'yield']).values
Y_train = train_data['yield'].values

no_of_features = X_train.shape[1]

test_data = pd.read_csv('data/test.csv')
X_test = test_data.drop(columns=['id']).values
ids_test = test_data['id']

In [3]:
#  Define the risk functions

def calculate_MSE(y_pred, y_true):
    """Calculate the mean squared error (MSE) as risk."""
    return np.mean((y_true - y_pred) ** 2)

def calculate_MAE(y_pred, y_true):
    """Calculate the mean absolute error (MAE) as risk."""
    return np.mean(np.abs(y_true - y_pred))

## Simple Gaussian KDE 

We have implemented a n-dimensional Gaussian Kernel, and are using it to estimate a good bandwidth using a 10-fold cross validation estimator and finally using kernel regression to get the estimated function

In [4]:
class GaussianKDE:
    def __init__(self, bandwidth=1.0):
        """Initialize the KDE model with the given bandwidth."""
        self.bandwidth = bandwidth
        self.data = None  # Training data matrix
        self.y_data = None  # Target values

    def fit(self, x_data, y_data):
        """Fit the KDE model with the given x and y data."""
        self.data = x_data  # Store feature matrix
        self.y_data = y_data  # Store target values
        self.dimension = x_data.shape[1]  # Dimension of input data

    def __gaussian_kernel(self, dists):
        """Gaussian kernel function for multivariate data."""
        exponent = -0.5 * np.sum(dists**2, axis=-1)  # Squared L2 norm of differences
        normalization_factor = (1 / ((2 * np.pi) ** (self.dimension / 2) * self.bandwidth ** self.dimension))  # Correct normalization factor
        return normalization_factor * np.exp(exponent)

    def gaussian_kernel(self, x, xi):
        """Apply Gaussian kernel for all points."""
        # Compute the distances for all points, using broadcasting.
        return self.__gaussian_kernel((x-xi)/self.bandwidth)

    def predict(self, x):
        func_val = []
        for point in x:
            weights = self.gaussian_kernel(point, self.data)
            numerator = np.sum(weights * self.y_data)
            denominator = np.sum(weights)
            
            if denominator > 0:
                func_val.append(numerator / denominator)
            else:
                func_val.append(0)  # Handle division by zero case
        return np.array(func_val)

In [5]:
def k_fold_KDE(kde, x_data, y_data, bandwidth, k:int = 10):
    
    data_length = int(len(x_data)/k)
    risk = 0
    for i in range(k):
        x_train = np.concatenate([x_data[:i*data_length], x_data[(i+1)*data_length:]])
        y_train = np.concatenate([y_data[:i*data_length], y_data[(i+1)*data_length:]])
        
        x_test = x_data[i*data_length:(i+1)*data_length]
        y_test = y_data[i*data_length:(i+1)*data_length]
        
        kernel = kde(bandwidth)
        kernel.fit(x_train, y_train)
        y_pred = kernel.predict(x_test)
        
        risk = risk + np.mean(np.abs(y_test - y_pred))
    
    risk = risk / k
    return risk

In [6]:
# # Used binary search to finally get a good value of bandwidth

# bandwidths_GaussianKDE = np.linspace(1.5, 1.75, 5)

# risks_Gkde = []
# for bandwidth in bandwidths_GaussianKDE:
#     risk = k_fold_KDE(GaussianKDE, X_train, Y_train, bandwidth)
#     risks_Gkde.append(risk)
#     print(f"bandwidth: {bandwidth}, risk: {risk}")

# best_bandwidth_Gkde = bandwidths_GaussianKDE[np.argmin(risks_Gkde)]
best_bandwidth_Gkde = 1.625
print(f"bandwidth corresponding to minimum estimated risk is {best_bandwidth_Gkde}")

bandwidth corresponding to minimum estimated risk is 1.625


In [7]:
# Use the calculated bandwidth to calculate the MAE on the train data 

kde = GaussianKDE(best_bandwidth_Gkde)
kde.fit(X_train, Y_train)
y_pred_train = kde.predict(X_train)
mae_train = calculate_MAE(y_pred_train, Y_train)
print(f"Mean Absolute Error on the train data: {mae_train}")

# MAE: 249

Mean Absolute Error on the train data: 249.22926102976695


## Simple linear multivariate OLS

Here, we use a simple linear multivariate regression to find the coefficients of each feature and use it to estimate the function.

In [8]:
class PolynomialOLS:
    def __init__(self,  degree: list[int]):
        self.degree = degree
        self.beta = None
        
    def __polynomial_matrix(self, X):
        """
        Generate polynomial features for multivariate data with individual degrees per feature.
        X: An array of shape (n_samples, n_features).
        Returns: A design matrix of shape (n_samples, n_polynomial_features) based on the degree list.
        """
        n_samples, n_features = X.shape
        X_poly = np.ones((n_samples, 1))
        
        if len(self.degree) != n_features:
            raise ValueError("Length of 'degree' must match the number of features in X.")

        # For each feature, generate polynomial features up to the specified degree for that feature
        for i in range(n_features):
            for d in range(1, self.degree[i] + 1):
                X_poly = np.hstack((X_poly, (X[:, i:i+1] ** d)))
        
        return X_poly
        
    def fit(self, X, Y):
        """
        Fit the model to the data using Ordinary Least Squares.
        X: An array of shape (n_samples, n_features).
        Y: The target values of shape (n_samples, 1).
        """
        X_poly = self.__polynomial_matrix(X)
        self.beta = np.linalg.inv(X_poly.T @ X_poly) @ (X_poly.T @ Y)
    
    def predict(self, X):
        """
        Predicts the values using the fitted model.
        X is the independent variable vector of shape (n, 1).
        Returns predicted values of shape (n, 1).
        """
        X_poly = self.__polynomial_matrix(X)
        return X_poly @ self.beta

In [9]:
def k_fold_SSR(polyOLS, x_data, y_data, degree, k:int = 10):
    
    data_length = int(len(x_data)/k)
    risk = 0
    for i in range(k):
        x_train = np.concatenate([x_data[:i*data_length], x_data[(i+1)*data_length:]])
        y_train = np.concatenate([y_data[:i*data_length], y_data[(i+1)*data_length:]])
        
        x_test = x_data[i*data_length:(i+1)*data_length]
        y_test = y_data[i*data_length:(i+1)*data_length]
        
        OLS = polyOLS(degree)
        OLS.fit(x_train, y_train)
        y_pred = OLS.predict(x_test)
        
        risk = risk + calculate_MAE(y_pred, y_test)
    risk = risk/k
    return risk

In [10]:
degree_all_ones = [1] * no_of_features

linearOLS1 = PolynomialOLS(degree_all_ones)
linearOLS1.fit(X_train, Y_train)
y_pred_train = linearOLS1.predict(X_train)
mae_train = calculate_MAE(y_pred_train, Y_train)
print(f"Mean Absolute Error on the train data: {mae_train}")

# MAE: 269

Mean Absolute Error on the train data: 269.5974461018767


## Linear multivariate OLS with dropping of features

Here, we find and drop the features on which our final estimate doesnt depend, and then use a simple linear multivariate regression to find the coefficients of each feature and use it to estimate the function.

In [11]:
def generate_next_degree_combinations_replace_1_with_0(current_best_degree):
    """
    Generate new degree combinations by introducing more 0s into the current best degree.
    Only place additional 0s where there are 1s in the current degree list.
    """
    next_combinations = []
    for i in range(len(current_best_degree)):
        if current_best_degree[i] == 1:
            new_combination = current_best_degree.copy()
            new_combination[i] = 0
            next_combinations.append(new_combination)
    return next_combinations

def iterative_degree_search_replace_1_with_0(X_train, Y_train, no_of_features, current_best_degree):
    # Start with the best combination from the first run
    best_ssr = np.inf
    converged = False
    
    while not converged:
        converged = True
        next_degree_combinations = generate_next_degree_combinations_replace_1_with_0(current_best_degree)
        SSR_at_degree = []
        
        # Evaluate SSR for each new combination
        for degree_list in next_degree_combinations:
            SSR_val = k_fold_SSR(PolynomialOLS, X_train, Y_train, degree_list)
            SSR_at_degree.append(SSR_val)
        
        # Find the combination with the lowest SSR in this round
        min_ssr = min(SSR_at_degree)
        best_degree_idx = np.argmin(np.array(SSR_at_degree))
        best_combination = next_degree_combinations[best_degree_idx]
        
        # Check if the new combination improves the SSR
        if min_ssr < best_ssr:
            best_ssr = min_ssr
            current_best_degree = best_combination
            converged = False  # Continue refining if SSR improves
            # print(f"Found better degree combination: {current_best_degree} with SSR: {best_ssr}")
        # else:
            # print(f"No further improvement. Current best combination: {current_best_degree} with SSR: {best_ssr}")
    
    print("The best degree combination found is:", current_best_degree)
    return current_best_degree

In [12]:
new_degree_estimate = iterative_degree_search_replace_1_with_0(X_train, Y_train, no_of_features, degree_all_ones)

linearOLS2 = PolynomialOLS(new_degree_estimate)
linearOLS2.fit(X_train, Y_train)
y_pred_train = linearOLS2.predict(X_train)
mae_train = calculate_MAE(y_pred_train, Y_train)
print(f"Mean Absolute Error on the train data: {mae_train}")

# MAE: 269

The best degree combination found is: [0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0, 1]
Mean Absolute Error on the train data: 269.33977712811406


## KDE with dropped columns

I ll assume that the columns which dont contribute in Linear regression dont contribute in KDE either

In [13]:
class GaussianKDE2:
    def __init__(self, degree: list[int], bandwidth=1.0):
        """Initialize the KDE model with the given bandwidth."""
        self.bandwidth = bandwidth
        self.degree = degree  # A list indicating which columns to use (1 means include, 0 means ignore)
        self.data = None  # Training data matrix
        self.y_data = None  # Target values

    def fit(self, x_data, y_data):
        """Fit the KDE model with the given x and y data."""
        # Select only the columns corresponding to degree == 1
        self.data = x_data[:, np.array(self.degree) == 1]  # Filter relevant columns based on degree
        self.y_data = y_data  # Store target values
        self.dimension = self.data.shape[1]  # Effective dimension based on selected features

    def __gaussian_kernel(self, dists):
        """Gaussian kernel function for multivariate data."""
        exponent = -0.5 * np.sum(dists**2, axis=-1)  # Squared L2 norm of differences
        normalization_factor = (1 / ((2 * np.pi) ** (self.dimension / 2) * self.bandwidth ** self.dimension))  # Correct normalization factor
        return normalization_factor * np.exp(exponent)

    def gaussian_kernel(self, x, xi):
        """Apply Gaussian kernel for all points."""
        # Select only the relevant columns from x based on the degree
        x_relevant = x[np.array(self.degree) == 1]  # Keep only selected features
        return self.__gaussian_kernel((x_relevant - xi) / self.bandwidth)

    def predict(self, x):
        """Evaluate the Nadaraya-Watson estimator at points x."""
        func_val = []
        
        for point in x:
            weights = self.gaussian_kernel(point, self.data)
            numerator = np.sum(weights * self.y_data)
            denominator = np.sum(weights)
            
            if denominator > 0:
                func_val.append(numerator / denominator)
            else:
                func_val.append(0)  # Handle division by zero case
        return np.array(func_val)

In [14]:
def k_fold_KDE2(kde, x_data, y_data, bandwidth, degree, k:int = 10):
    
    data_length = int(len(x_data)/k)
    risk = 0
    for i in range(k):
        x_train = np.concatenate([x_data[:i*data_length], x_data[(i+1)*data_length:]])
        y_train = np.concatenate([y_data[:i*data_length], y_data[(i+1)*data_length:]])
        
        x_test = x_data[i*data_length:(i+1)*data_length]
        y_test = y_data[i*data_length:(i+1)*data_length]
        
        kernel = kde(degree, bandwidth)
        kernel.fit(x_train, y_train)
        y_pred = kernel.predict(x_test)
        
        risk = risk + np.mean(np.abs(y_test - y_pred))
    
    risk = risk / k
    return risk

In [15]:
# # Used binary search to finally get a good value of bandwidth

# bandwidths_GaussianKDE2 = np.linspace(0.1, 0.2, 10)

# risks_Gkde = []
# for bandwidth in bandwidths_GaussianKDE2:
#     risk = k_fold_KDE2(GaussianKDE2, X_train, Y_train, bandwidth, new_degree_estimate)
#     risks_Gkde.append(risk)
#     print(f"bandwidth: {bandwidth}, risk: {risk}")

# best_bandwidth_Gkde2 = bandwidths_GaussianKDE2[np.argmin(risks_Gkde)]
best_bandwidth_Gkde2 = 0.1667
print(f"bandwidth corresponding to minimum estimated risk is {best_bandwidth_Gkde2}")

bandwidth corresponding to minimum estimated risk is 0.1667


In [16]:
# Use the calculated bandwidth to calculate the MAE on the train data 

kde = GaussianKDE2(new_degree_estimate, best_bandwidth_Gkde2)
kde.fit(X_train, Y_train)
y_pred_train = kde.predict(X_train)
mae_train = calculate_MAE(y_pred_train, Y_train)
print(f"Mean Absolute Error on the train data: {mae_train}")

# MAE: 303

Mean Absolute Error on the train data: 303.4226201137313


## Non-Linear multivariate OLS with dropping of features

Here, we find and drop the features on which our final estimate doesnt depend, and then find the features on which a quadratic fit is better, and use LS regression to find the coefficients of each feature and use it to estimate the function.

In [17]:
def generate_next_degree_combinations_replace_1_with_2(current_best_degree):
    """
    Generate new degree combinations by replacing 1s with 2s in the current best degree list.
    Only change degrees that are currently 1.
    """
    next_combinations = []
    for i in range(len(current_best_degree)):
        if current_best_degree[i] == 1:
            new_combination = current_best_degree.copy()
            new_combination[i] = 2
            next_combinations.append(new_combination)
    return next_combinations

def iterative_degree_search_replace_1_with_2(X_train, Y_train, no_of_features, current_best_degree):
    best_ssr = np.inf
    converged = False
    
    while not converged:
        converged = True
        next_degree_combinations = generate_next_degree_combinations_replace_1_with_2(current_best_degree)
        SSR_at_degree = []
        
        # Evaluate SSR for each new combination
        for degree_list in next_degree_combinations:
            SSR_val = k_fold_SSR(PolynomialOLS, X_train, Y_train, degree_list)
            SSR_at_degree.append(SSR_val)
        
        # Find the combination with the lowest SSR in this round
        min_ssr = min(SSR_at_degree)
        best_degree_idx = np.argmin(np.array(SSR_at_degree))
        best_combination = next_degree_combinations[best_degree_idx]
        
        # Check if the new combination improves the SSR
        if min_ssr < best_ssr:
            best_ssr = min_ssr
            current_best_degree = best_combination
            converged = False  # Continue refining if SSR improves
        #     print(f"Found better degree combination: {current_best_degree} with SSR: {best_ssr}")
        # else:
        #     print(f"No further improvement. Current best combination: {current_best_degree} with SSR: {best_ssr}")
    
    return current_best_degree

In [18]:
new_degree_estimate2 = iterative_degree_search_replace_1_with_2(X_train, Y_train, no_of_features, new_degree_estimate)

linearOLS3 = PolynomialOLS(new_degree_estimate2)
linearOLS3.fit(X_train, Y_train)
y_pred_train = linearOLS3.predict(X_train)
mae_train = calculate_MAE(y_pred_train, Y_train)
print(f"Mean Absolute Error on the train data: {mae_train}")

# MAE: 269

Mean Absolute Error on the train data: 269.2946045204728


Concluding, the least MAE is obtained using a normal linear regression considering all feautures

In [19]:
y_pred = linearOLS1.predict(X_test)

predictions_df = pd.DataFrame({
    'id': ids_test,
    'y': y_pred.flatten()
})
predictions_df.to_csv("submission.csv", index=False)